# 🏦 Enterprise Mock Data Loader

**Generates 3 years of realistic Databricks cost & usage data** modeled after large financial institutions (Chase, BOFA scale):

| Metric | Value |
|--------|-------|
| Annual spend forecast | ~$5M |
| Active users | 1,100+ |
| Workspaces | 5 LOBs |
| Clusters | 200 |
| SQL Warehouses | 30 |
| Jobs | 500 |
| ML Experiments | 20 |
| DLT Pipelines | 20 |
| Serving Endpoints | 8 |

**Run all cells below** — takes ~15-30 min depending on warehouse size.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
# If running in Databricks notebook, these are auto-injected.
# If running locally, set environment variables.

import os, sys

# Try Databricks notebook context first
try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    HOST = ctx.apiUrl().get()
    TOKEN = ctx.apiToken().get()
    print(f'Running in Databricks notebook: {HOST}')
except:
    HOST = os.environ.get('DATABRICKS_HOST', '')
    TOKEN = os.environ.get('DATABRICKS_TOKEN', '')
    print(f'Running locally: {HOST}')

WAREHOUSE_ID = os.environ.get('DATABRICKS_WAREHOUSE_ID', '21f5bd20b7f44a51')
print(f'Warehouse: {WAREHOUSE_ID}')

In [ ]:
# ── SQL Execution Helper ──────────────────────────────────────────────
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementState

client = WorkspaceClient(host=HOST, token=TOKEN)

def run_sql(label, sql, timeout_sec=300):
    """Execute SQL and wait for completion."""
    resp = client.statement_execution.execute_statement(
        warehouse_id=WAREHOUSE_ID, statement=sql, wait_timeout='50s')
    deadline = time.time() + timeout_sec
    while resp.status.state in (StatementState.PENDING, StatementState.RUNNING):
        if time.time() > deadline:
            print(f'  ⏱ TIMEOUT: {label}')
            return False
        time.sleep(3)
        resp = client.statement_execution.get_statement(resp.statement_id)
    if resp.status.state == StatementState.SUCCEEDED:
        print(f'  ✅ {label}')
        return True
    err = getattr(resp.status.error, 'message', str(resp.status.state))
    print(f'  ⚠️ SKIP ({err[:100]}): {label}')
    return False

print('SQL executor ready')

In [ ]:
# ── Constants & Data Generation ───────────────────────────────────────
import random
from datetime import datetime, timedelta

ACCOUNT_ID = 'acc-chase-001'
WORKSPACES = [
    ('ws-100001', 'prod-consumer-banking', 'us-east-1'),
    ('ws-100002', 'prod-investment-banking', 'us-east-1'),
    ('ws-100003', 'prod-risk-analytics', 'us-west-2'),
    ('ws-100004', 'prod-data-platform', 'us-east-1'),
    ('ws-100005', 'prod-fraud-detection', 'us-west-2'),
]

DEPARTMENTS = [
    'Consumer Banking', 'Investment Banking', 'Risk Analytics',
    'Fraud Detection', 'Data Engineering', 'Data Science',
    'Compliance', 'Treasury', 'Marketing Analytics',
    'Credit Risk', 'Market Risk', 'Operations',
    'Wealth Management', 'Card Services', 'Mortgage',
]

TEAMS = [
    'platform-core', 'etl-pipeline', 'ml-ops', 'analytics-eng',
    'feature-store', 'data-quality', 'streaming-ingest',
    'reporting', 'bi-team', 'quant-research', 'fraud-ml',
    'aml-detection', 'credit-scoring', 'nrt-analytics',
    'data-governance', 'lakehouse-admin', 'cost-optimization',
]

SKU_MAP = {
    'STANDARD_ALL_PURPOSE_COMPUTE': 0.55, 'PREMIUM_ALL_PURPOSE_COMPUTE': 0.70,
    'JOBS_COMPUTE': 0.15, 'JOBS_LIGHT_COMPUTE': 0.10,
    'SERVERLESS_SQL': 0.70, 'PRO_SQL': 0.55,
    'SERVERLESS_REAL_TIME_INFERENCE': 0.07, 'GPU_ALL_PURPOSE_COMPUTE': 1.50,
    'DLT_CORE_COMPUTE': 0.20, 'DLT_PRO_COMPUTE': 0.25, 'DLT_ADVANCED_COMPUTE': 0.36,
}

SKU_WEIGHTS = {
    'JOBS_COMPUTE': 0.35, 'SERVERLESS_SQL': 0.20, 'PRO_SQL': 0.10,
    'STANDARD_ALL_PURPOSE_COMPUTE': 0.08, 'PREMIUM_ALL_PURPOSE_COMPUTE': 0.05,
    'DLT_PRO_COMPUTE': 0.07, 'DLT_ADVANCED_COMPUTE': 0.04, 'DLT_CORE_COMPUTE': 0.03,
    'GPU_ALL_PURPOSE_COMPUTE': 0.04, 'SERVERLESS_REAL_TIME_INFERENCE': 0.02,
    'JOBS_LIGHT_COMPUTE': 0.02,
}

NODE_TYPES = [
    'i3.xlarge', 'i3.2xlarge', 'i3.4xlarge', 'i3.8xlarge',
    'r5.xlarge', 'r5.2xlarge', 'r5.4xlarge', 'r5.8xlarge',
    'm5.xlarge', 'm5.2xlarge', 'm5.4xlarge',
    'p3.2xlarge', 'p3.8xlarge', 'g4dn.xlarge', 'g4dn.4xlarge',
]

DBR_VERSIONS = [
    '12.2.x-scala2.12', '13.0.x-scala2.12', '13.3.x-scala2.12',
    '14.0.x-scala2.12', '14.3.x-scala2.12', '15.0.x-scala2.12',
    '15.2.x-scala2.12', '15.4.x-scala2.12',
]

FIRST_NAMES = [
    'James','Mary','Robert','Patricia','John','Jennifer','Michael','Linda','David','Elizabeth',
    'William','Barbara','Richard','Susan','Joseph','Jessica','Thomas','Sarah','Christopher','Karen',
    'Charles','Lisa','Daniel','Nancy','Matthew','Betty','Anthony','Margaret','Mark','Sandra',
    'Donald','Ashley','Steven','Dorothy','Paul','Kimberly','Andrew','Emily','Joshua','Donna',
    'Kenneth','Michelle','Kevin','Carol','Brian','Amanda','George','Melissa','Timothy','Deborah',
    'Ronald','Stephanie','Edward','Rebecca','Jason','Sharon','Jeffrey','Laura','Ryan','Cynthia',
    'Jacob','Kathleen','Gary','Amy','Nicholas','Angela','Eric','Shirley','Jonathan','Anna',
    'Stephen','Brenda','Larry','Pamela','Justin','Emma','Scott','Nicole','Brandon','Helen',
    'Benjamin','Samantha','Samuel','Katherine','Raymond','Christine','Gregory','Debra','Frank','Rachel',
    'Alexander','Carolyn','Patrick','Janet','Jack','Catherine','Dennis','Maria','Jerry','Heather',
    'Tyler','Diane','Aaron','Ruth','Jose','Julie','Nathan','Olivia','Henry','Joyce',
    'Peter','Virginia','Douglas','Victoria','Zachary','Kelly','Kyle','Lauren','Noah','Christina',
    'Ethan','Joan','Adrian','Evelyn','Aiden','Judith','Dylan','Megan',
    'Priya','Raj','Anita','Vikram','Deepa','Sanjay','Neha','Amit',
    'Wei','Ming','Yuki','Hiro','Jin','Soo','Chen','Li',
]

LAST_NAMES = [
    'Smith','Johnson','Williams','Brown','Jones','Garcia','Miller','Davis','Rodriguez','Martinez',
    'Hernandez','Lopez','Gonzalez','Wilson','Anderson','Thomas','Taylor','Moore','Jackson','Martin',
    'Lee','Perez','Thompson','White','Harris','Sanchez','Clark','Ramirez','Lewis','Robinson',
    'Walker','Young','Allen','King','Wright','Scott','Torres','Nguyen','Hill','Flores','Green',
    'Adams','Nelson','Baker','Hall','Rivera','Campbell','Mitchell','Carter','Roberts',
    'Patel','Shah','Kumar','Singh','Gupta','Sharma','Chen','Wang','Li','Zhang','Liu','Yang','Wu',
    'Kim','Park','Cho','Jung','Tanaka','Suzuki','Watanabe',
    "O'Brien",'Murphy','Sullivan','Cohen','Goldberg','Katz',
]

def sql_str(s):
    return s.replace("'", "''")

def chunk_list(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

# Generate users
def generate_users(count=1100):
    users, used = [], set()
    random.seed(42)
    for _ in range(count * 2):
        if len(users) >= count: break
        email = f"{random.choice(FIRST_NAMES).lower()}.{random.choice(LAST_NAMES).lower()}@chase.com"
        if email not in used:
            used.add(email); users.append(email)
    return users

USERS = generate_users(1100)
NOW = datetime(2026, 4, 26)
THREE_YEARS_AGO = NOW - timedelta(days=1095)

random.seed(42)
USER_DEPT = {u: random.choice(DEPARTMENTS) for u in USERS}
USER_TEAM = {u: random.choice(TEAMS) for u in USERS}

print(f'Generated {len(USERS)} users')
print(f'Date range: {THREE_YEARS_AGO.strftime("%Y-%m-%d")} → {NOW.strftime("%Y-%m-%d")}')

In [ ]:
# ── Generate Cluster, Warehouse, Job definitions ─────────────────────

def generate_clusters(count=200):
    random.seed(100)
    clusters = []
    for i in range(count):
        ws = random.choice(WORKSPACES)
        owner = random.choice(USERS)
        cname = '-'.join([random.choice(['etl','analytics','ml','streaming','adhoc','prod','staging','dev']),
                          random.choice(['pipeline','cluster','compute','workload','processing']),
                          str(random.randint(1,50))])
        clusters.append(dict(
            ws=ws, id=f'cls-{i+1:04d}', name=cname, owner=owner,
            driver=random.choice(NODE_TYPES[:11]), worker=random.choice(NODE_TYPES[:11]),
            workers=random.choice([2,4,8,16,32]),
            min_w=max(1, random.choice([2,4,8,16,32])//4),
            max_w=random.choice([2,4,8,16,32])*2,
            auto_term=random.choice([60,120,240,0]),
            dbr=random.choice(DBR_VERSIONS), team=USER_TEAM[owner], dept=USER_DEPT[owner],
            days_ago=random.randint(30,1000), deleted=random.random()<0.15,
            security=random.choice(['SINGLE_USER','USER_ISOLATION','NO_ISOLATION'])))
    return clusters

def generate_warehouses(count=30):
    random.seed(200)
    whs = []
    sizes = ['2X-Small','X-Small','Small','Medium','Large','X-Large','2X-Large']
    for i in range(count):
        ws = random.choice(WORKSPACES)
        whs.append(dict(
            ws=ws, id=f'wh-{i+1:04d}',
            name=f"{random.choice(['reporting','bi','adhoc','etl','prod','staging'])}-warehouse-{i+1}",
            type=random.choice(['PRO','CLASSIC','SERVERLESS']),
            size=random.choice(sizes), min_c=random.choice([1,1,1,2]),
            max_c=random.choice([1,2,4,8,16]), auto_stop=random.choice([5,10,15,30]),
            days_ago=random.randint(30,900)))
    return whs

def generate_jobs(count=500):
    random.seed(300)
    jobs = []
    job_types = ['ETL-Daily','ETL-Hourly','ML-Training','ML-Scoring','Report-Generation',
                 'Data-Quality','Feature-Engineering','Streaming-Ingest','CDC-Pipeline',
                 'Archive-Job','Compliance-Check','AML-Scan','Fraud-Score','Risk-Calc',
                 'PnL-Report','Regulatory-Filing','Customer-360','Segmentation',
                 'Campaign-Analytics','Real-Time-Alerts']
    schedules = ['0 0 * * * ?','0 0 8 * * ?','0 0 6 * * ?','0 30 7 * * ?',
                 '0 0 0 * * ?','0 0 */4 * * ?','0 0 9 ? * MON']
    for i in range(count):
        ws = random.choice(WORKSPACES)
        creator = random.choice(USERS)
        jtype = random.choice(job_types)
        jobs.append(dict(
            ws=ws, id=f'job-{i+1:05d}',
            name=f"{jtype}-{USER_DEPT[creator].lower().replace(' ','-')}-{random.randint(1,99)}",
            creator=creator, schedule=random.choice(schedules),
            days_ago=random.randint(10,1000), deleted=random.random()<0.08,
            team=USER_TEAM[creator], env=random.choice(['prod','staging','dev'])))
    return jobs

CLUSTERS = generate_clusters()
WAREHOUSES = generate_warehouses()
JOBS = generate_jobs()

print(f'Clusters: {len(CLUSTERS)}, Warehouses: {len(WAREHOUSES)}, Jobs: {len(JOBS)}')

In [ ]:
# ── Step 1: Create all schemas ───────────────────────────────────────
print('Creating schemas...')
for schema in ['billing','access','compute','lakeflow','query','ai_gateway',
               'serving','mlflow','storage','information_schema',
               'networking','lakeview','dashboards','marketplace']:
    run_sql(f'schema {schema}', f'CREATE SCHEMA IF NOT EXISTS workspace.mock_system_{schema}')
print('Done!')

In [ ]:
# ── Step 2: billing.usage (3 years, ~$5M/year) ──────────────────────
print('Generating billing usage data (3 years)...')

run_sql('drop billing.usage', 'DROP TABLE IF EXISTS workspace.mock_system_billing.usage')
run_sql('create billing.usage', '''
CREATE TABLE workspace.mock_system_billing.usage (
  record_id STRING, account_id STRING, workspace_id STRING, sku_name STRING, cloud STRING,
  usage_start_time TIMESTAMP, usage_end_time TIMESTAMP, usage_date DATE,
  custom_tags MAP<STRING, STRING>, usage_unit STRING, usage_quantity DOUBLE,
  usage_type STRING, billing_origin_product STRING, record_type STRING, ingestion_date DATE,
  identity_metadata STRUCT<run_as: STRING, created_by: STRING>,
  usage_metadata STRUCT<cluster_id: STRING, warehouse_id: STRING, job_id: STRING,
    job_run_id: STRING, dlt_pipeline_id: STRING, notebook_id: STRING,
    endpoint_name: STRING, endpoint_id: STRING, run_id: STRING>
)
''')

random.seed(42)
rows = []
rc = 0
yearly_daily_target = {0: 9600, 1: 12300, 2: 15000}
current = THREE_YEARS_AGO

while current < NOW:
    yi = min(2, (current - THREE_YEARS_AGO).days // 365)
    dt = yearly_daily_target[yi]
    dow = current.weekday()
    if dow >= 5: dt = int(dt * 0.4)
    if current.day >= 28: dt = int(dt * 1.3)
    if current.month in (3,6,9,12) and current.day >= 25: dt = int(dt * 1.5)

    for sku, weight in SKU_WEIGHTS.items():
        sb = int(dt * weight)
        if sb < 1: continue
        price = SKU_MAP[sku]
        total_dbu = sb / price
        nr = random.randint(5, 25)
        for _ in range(nr):
            rc += 1
            user = random.choice(USERS)
            ws = random.choice(WORKSPACES)
            team = USER_TEAM[user]; dept = USER_DEPT[user]
            dbu = round(total_dbu / nr * random.uniform(0.5, 1.5), 2)
            ho = random.randint(0, 23); dm = random.randint(10, 180)
            us = current.replace(hour=ho, minute=0, second=0)
            ue = us + timedelta(minutes=dm)
            cid=wid=jid=jrid=dltid=nbid=epn=epi=rid=''
            if 'ALL_PURPOSE' in sku or 'GPU' in sku:
                cid = random.choice(CLUSTERS)['id']; nbid = f'nb-{random.randint(1,5000)}'
            elif 'SQL' in sku: wid = random.choice(WAREHOUSES)['id']
            elif 'JOBS' in sku: j=random.choice(JOBS); jid=j['id']; jrid=f'run-{rc}'
            elif 'DLT' in sku: dltid = f'dlt-{random.randint(1,100):03d}'
            elif 'INFERENCE' in sku:
                epn = random.choice(['fraud-scorer','credit-model','recommender','nlp-classifier'])
                epi = f'ep-{random.randint(1,20):03d}'
            bp = 'INTERACTIVE' if 'ALL_PURPOSE' in sku else ('SQL' if 'SQL' in sku else ('JOBS' if 'JOBS' in sku else ('DLT' if 'DLT' in sku else 'SERVING')))
            rows.append(
                f"('r-{rc:08d}','{ACCOUNT_ID}','{ws[0]}','{sku}','AWS',"
                f"'{us.strftime('%Y-%m-%d %H:%M:%S')}','{ue.strftime('%Y-%m-%d %H:%M:%S')}',"
                f"'{current.strftime('%Y-%m-%d')}',map('team','{sql_str(team)}','department','{sql_str(dept)}'),'DBU',{dbu},'{bp}',"
                f"'{bp}','ORIGINAL','{current.strftime('%Y-%m-%d')}',"
                f"named_struct('run_as','{sql_str(user)}','created_by','{sql_str(user)}'),"
                f"named_struct('cluster_id','{cid}','warehouse_id','{wid}','job_id','{jid}','job_run_id','{jrid}',"
                f"'dlt_pipeline_id','{dltid}','notebook_id','{nbid}','endpoint_name','{epn}','endpoint_id','{epi}','run_id','{rid}'))")
    current += timedelta(days=1)

print(f'Generated {len(rows)} billing rows. Inserting in chunks...')
for i, chunk in enumerate(chunk_list(rows, 500)):
    vals = ',\n'.join(chunk)
    run_sql(f'billing chunk {i+1}/{(len(rows)//500)+1}',
            f'INSERT INTO workspace.mock_system_billing.usage VALUES\n{vals}')
print(f'✅ billing.usage loaded: {len(rows)} rows')

In [ ]:
# ── Step 3: billing.list_prices ──────────────────────────────────────
run_sql('drop list_prices', 'DROP TABLE IF EXISTS workspace.mock_system_billing.list_prices')
run_sql('create list_prices', '''
CREATE TABLE workspace.mock_system_billing.list_prices (
  price_start_time TIMESTAMP, price_end_time TIMESTAMP, account_id STRING,
  sku_name STRING, cloud STRING, currency_code STRING, usage_unit STRING,
  pricing STRUCT<default: DOUBLE, promotional: DOUBLE, effective_list: DOUBLE>
)
''')
price_rows = []
for sku, price in SKU_MAP.items():
    price_rows.append(
        f"('{THREE_YEARS_AGO.strftime('%Y-%m-%d %H:%M:%S')}',cast(null as TIMESTAMP),"
        f"'{ACCOUNT_ID}','{sku}','AWS','USD','DBU',"
        f"named_struct('default',{price},'promotional',cast(null as DOUBLE),'effective_list',{price}))")
run_sql('insert list_prices', f"INSERT INTO workspace.mock_system_billing.list_prices VALUES\n" + ',\n'.join(price_rows))
print('✅ billing.list_prices loaded')

In [ ]:
# ── Step 4: access.workspaces_latest ─────────────────────────────────
run_sql('drop workspaces', 'DROP TABLE IF EXISTS workspace.mock_system_access.workspaces_latest')
run_sql('create workspaces', '''
CREATE TABLE workspace.mock_system_access.workspaces_latest (
  account_id STRING, workspace_id STRING, workspace_name STRING, workspace_url STRING,
  workspace_status STRING, cloud STRING, region STRING, pricing_tier STRING, creation_time TIMESTAMP
)
''')
ws_rows = [f"('{ACCOUNT_ID}','{wid}','{wn}','https://{wn}.cloud.databricks.com','RUNNING','AWS','{wr}','ENTERPRISE','{THREE_YEARS_AGO.strftime('%Y-%m-%d %H:%M:%S')}')" for wid,wn,wr in WORKSPACES]
run_sql('insert workspaces', f"INSERT INTO workspace.mock_system_access.workspaces_latest VALUES\n" + ',\n'.join(ws_rows))
print('✅ access.workspaces_latest loaded')

In [ ]:
# ── Step 5: access.audit (3 years) ───────────────────────────────────
print('Generating audit log data (3 years)...')

run_sql('drop audit', 'DROP TABLE IF EXISTS workspace.mock_system_access.audit')
run_sql('create audit', '''
CREATE TABLE workspace.mock_system_access.audit (
  account_id STRING, workspace_id STRING, version STRING, event_time TIMESTAMP,
  event_date DATE, source_ip_address STRING, user_agent STRING, session_id STRING,
  user_identity STRUCT<email: STRING, subjectName: STRING>,
  service_name STRING, action_name STRING, request_id STRING,
  request_params MAP<STRING, STRING>,
  response STRUCT<statusCode: INT, errorMessage: STRING, result: STRING>,
  audit_level STRING, event_id STRING,
  identity_metadata STRUCT<run_by: STRING, run_as: STRING>
)
''')

random.seed(55)
actions = [
    ('accounts','login'),('accounts','logout'),('clusters','create'),('clusters','start'),
    ('clusters','terminate'),('jobs','create'),('jobs','runNow'),('sql','commandSubmit'),
    ('sql','commandFinish'),('notebook','runCommand'),('databrickssql','getWarehouse'),
    ('secrets','getSecret'),('unityCatalog','getTable'),('unityCatalog','createTable'),
    ('mlflow','createRun'),('workspace','fileCreate'),('iamRole','changePermissions'),
]
audit_rows = []
ctr = 0
cur = THREE_YEARS_AGO
while cur < NOW:
    for _ in range(random.randint(30, 80)):
        ctr += 1
        user = random.choice(USERS); ws = random.choice(WORKSPACES)
        svc, act = random.choice(actions)
        et = cur.replace(hour=random.randint(6,22), minute=random.randint(0,59), second=random.randint(0,59))
        st = 200 if random.random() < 0.95 else random.choice([401,403,500])
        em = '' if st == 200 else 'Access denied'
        res = 'success' if st == 200 else 'failure'
        ip = f'{random.randint(10,172)}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}'
        err_sql = 'cast(null as STRING)' if not em else repr(em)
        audit_rows.append(
            f"('{ACCOUNT_ID}','{ws[0]}','2.0','{et.strftime('%Y-%m-%d %H:%M:%S')}','{cur.strftime('%Y-%m-%d')}',"
            f"'{ip}','Databricks/API','sess-{ctr:08d}',"
            f"named_struct('email','{sql_str(user)}','subjectName','{sql_str(user)}'),"
            f"'{svc}','{act}','req-{ctr:08d}',map('user','{sql_str(user)}'),"
            f"named_struct('statusCode',{st},'errorMessage',{err_sql},'result','{res}'),"
            f"'WORKSPACE_LEVEL','evt-{ctr:08d}',"
            f"named_struct('run_by','{sql_str(user)}','run_as','{sql_str(user)}'))")
    cur += timedelta(days=1)

print(f'Generated {len(audit_rows)} audit rows. Inserting...')
for i, chunk in enumerate(chunk_list(audit_rows, 500)):
    run_sql(f'audit chunk {i+1}/{(len(audit_rows)//500)+1}', f'INSERT INTO workspace.mock_system_access.audit VALUES\n' + ',\n'.join(chunk))
print(f'✅ access.audit loaded: {len(audit_rows)} rows')

In [ ]:
# ── Step 6: compute.clusters + warehouses ────────────────────────────
run_sql('drop clusters', 'DROP TABLE IF EXISTS workspace.mock_system_compute.clusters')
run_sql('create clusters', '''
CREATE TABLE workspace.mock_system_compute.clusters (
  account_id STRING, workspace_id STRING, cluster_id STRING, cluster_name STRING,
  owned_by STRING, create_time TIMESTAMP, delete_time TIMESTAMP,
  driver_node_type STRING, worker_node_type STRING, worker_count BIGINT,
  min_autoscale_workers BIGINT, max_autoscale_workers BIGINT,
  auto_termination_minutes BIGINT, enable_elastic_disk BOOLEAN,
  tags MAP<STRING, STRING>, cluster_source STRING, dbr_version STRING,
  change_time TIMESTAMP, change_date DATE, data_security_mode STRING
)
''')

cl_rows = []
for c in CLUSTERS:
    ws = c['ws']
    cd = (NOW - timedelta(days=c['days_ago'])).strftime('%Y-%m-%d %H:%M:%S')
    dt_str = f"'{(NOW - timedelta(days=random.randint(1, c['days_ago']))).strftime('%Y-%m-%d %H:%M:%S')}'" if c['deleted'] else 'cast(null as TIMESTAMP)'
    chd = (NOW - timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d')
    cl_rows.append(
        f"('{ACCOUNT_ID}','{ws[0]}','{c['id']}','{c['name']}','{sql_str(c['owner'])}',"
        f"'{cd}',{dt_str},'{c['driver']}','{c['worker']}',{c['workers']},{c['min_w']},{c['max_w']},"
        f"{c['auto_term']},true,map('team','{c['team']}','department','{c['dept']}'),'API','{c['dbr']}',"
        f"'{chd} 10:00:00','{chd}','{c['security']}')")

run_sql('insert clusters', f'INSERT INTO workspace.mock_system_compute.clusters VALUES\n' + ',\n'.join(cl_rows))

# Warehouses
run_sql('drop warehouses', 'DROP TABLE IF EXISTS workspace.mock_system_compute.warehouses')
run_sql('create warehouses', '''
CREATE TABLE workspace.mock_system_compute.warehouses (
  account_id STRING, workspace_id STRING, warehouse_id STRING, warehouse_name STRING,
  warehouse_type STRING, warehouse_size STRING, min_clusters INT, max_clusters INT,
  auto_stop_minutes INT, change_time TIMESTAMP, delete_time TIMESTAMP
)
''')
wh_rows = []
for wh in WAREHOUSES:
    ws = wh['ws']
    ch = (NOW - timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d %H:%M:%S')
    wh_rows.append(
        f"('{ACCOUNT_ID}','{ws[0]}','{wh['id']}','{wh['name']}','{wh['type']}','{wh['size']}',"
        f"{wh['min_c']},{wh['max_c']},{wh['auto_stop']},'{ch}',cast(null as TIMESTAMP))")
run_sql('insert warehouses', f'INSERT INTO workspace.mock_system_compute.warehouses VALUES\n' + ',\n'.join(wh_rows))

# Empty event tables
for tbl, ddl in [
    ('cluster_events', 'account_id STRING,workspace_id STRING,cluster_id STRING,timestamp TIMESTAMP,type STRING,details MAP<STRING,STRING>'),
    ('warehouse_events', 'account_id STRING,workspace_id STRING,warehouse_id STRING,event_type STRING,cluster_count INT,event_time TIMESTAMP'),
]:
    run_sql(f'drop {tbl}', f'DROP TABLE IF EXISTS workspace.mock_system_compute.{tbl}')
    run_sql(f'create {tbl}', f'CREATE TABLE workspace.mock_system_compute.{tbl} ({ddl})')

print(f'✅ compute tables loaded: {len(CLUSTERS)} clusters, {len(WAREHOUSES)} warehouses')

In [ ]:
# ── Step 7: compute.node_timeline (90 days) ─────────────────────────
print('Generating node timeline (90 days)...')
run_sql('drop node_timeline', 'DROP TABLE IF EXISTS workspace.mock_system_compute.node_timeline')
run_sql('create node_timeline', '''
CREATE TABLE workspace.mock_system_compute.node_timeline (
  account_id STRING, workspace_id STRING, cluster_id STRING, node_id STRING,
  instance_id STRING, start_time TIMESTAMP, end_time TIMESTAMP,
  driver BOOLEAN, is_driver BOOLEAN,
  cpu_user_percent DOUBLE, cpu_system_percent DOUBLE, cpu_wait_percent DOUBLE,
  mem_used_percent DOUBLE, mem_swap_percent DOUBLE,
  network_sent_bytes BIGINT, network_received_bytes BIGINT,
  node_type STRING, private_ip STRING, uptime_seconds DOUBLE,
  num_task_slots INT, avg_num_running_tasks DOUBLE, avg_num_queued_tasks DOUBLE
)
''')

random.seed(111)
node_rows = []
for day_off in range(90):
    cd = NOW - timedelta(days=day_off)
    for cl in random.sample(CLUSTERS, min(40, len(CLUSTERS))):
        ws = cl['ws']
        for ni in range(random.randint(2, cl['workers']+1)):
            isd = (ni == 0)
            cpu = random.uniform(5,95); cpus = random.uniform(1,15)
            mem = random.uniform(20,95)
            st = cd.replace(hour=random.randint(6,22), minute=0, second=0)
            en = st + timedelta(hours=random.randint(1,8))
            node_rows.append(
                f"('{ACCOUNT_ID}','{ws[0]}','{cl['id']}','node-{day_off:03d}-{cl['id']}-{ni}',"
                f"'i-{random.randint(10000,99999):05d}{random.randint(10000,99999):05d}',"
                f"'{st.strftime('%Y-%m-%d %H:%M:%S')}','{en.strftime('%Y-%m-%d %H:%M:%S')}',"
                f"{str(isd).lower()},{str(isd).lower()},{cpu:.1f},{cpus:.1f},{random.uniform(0,10):.1f},"
                f"{mem:.1f},{random.uniform(0,2):.1f},{random.randint(100000,10000000000)},"
                f"{random.randint(100000,10000000000)},'{cl['worker']}',"
                f"'10.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}',"
                f"{random.uniform(3600,28800):.0f},{random.randint(4,32)},"
                f"{random.uniform(0.5,28):.1f},{random.uniform(0,5):.1f})")

print(f'Generated {len(node_rows)} node rows. Inserting...')
for i, chunk in enumerate(chunk_list(node_rows, 500)):
    run_sql(f'nodes chunk {i+1}', f'INSERT INTO workspace.mock_system_compute.node_timeline VALUES\n' + ',\n'.join(chunk))
print(f'✅ compute.node_timeline loaded: {len(node_rows)} rows')

In [ ]:
# ── Step 8: lakeflow.jobs + job_run_timeline (3 years) ───────────────
print('Loading lakeflow tables...')

run_sql('drop jobs', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.jobs')
run_sql('create jobs', '''
CREATE TABLE workspace.mock_system_lakeflow.jobs (
  account_id STRING, workspace_id STRING, job_id STRING, name STRING,
  creator_user_name STRING, run_as_user_name STRING, tags MAP<STRING, STRING>,
  schedule STRUCT<quartz_cron_expression: STRING, pause_status: STRING>,
  created_time TIMESTAMP, change_time TIMESTAMP, delete_time TIMESTAMP
)
''')
j_rows = []
for j in JOBS:
    ws = j['ws']
    cr = (NOW - timedelta(days=j['days_ago'])).strftime('%Y-%m-%d %H:%M:%S')
    ch = (NOW - timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d %H:%M:%S')
    dt = f"'{(NOW - timedelta(days=random.randint(1,j['days_ago']))).strftime('%Y-%m-%d %H:%M:%S')}'" if j['deleted'] else 'cast(null as TIMESTAMP)'
    pa = 'UNPAUSED' if random.random()<0.85 else 'PAUSED'
    j_rows.append(
        f"('{ACCOUNT_ID}','{ws[0]}','{j['id']}','{sql_str(j['name'])}',"
        f"'{sql_str(j['creator'])}','{sql_str(j['creator'])}',"
        f"map('Env','{j['env']}','team','{j['team']}'),"
        f"named_struct('quartz_cron_expression','{j['schedule']}','pause_status','{pa}'),"
        f"'{cr}','{ch}',{dt})")
for i, chunk in enumerate(chunk_list(j_rows, 200)):
    run_sql(f'jobs chunk {i+1}', f'INSERT INTO workspace.mock_system_lakeflow.jobs VALUES\n' + ',\n'.join(chunk))

# Job run timeline
print('Generating job run timeline (3 years)...')
run_sql('drop job_runs', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.job_run_timeline')
run_sql('create job_runs', '''
CREATE TABLE workspace.mock_system_lakeflow.job_run_timeline (
  account_id STRING, workspace_id STRING, job_id STRING, run_id STRING,
  period_start_time TIMESTAMP, period_end_time TIMESTAMP,
  trigger_type STRING, run_type STRING, result_state STRING, termination_code STRING
)
''')

random.seed(88)
jr_rows = []
jrc = 0
cur = THREE_YEARS_AGO
while cur < NOW:
    for _ in range(random.randint(80, 300)):
        jrc += 1
        j = random.choice(JOBS); ws = j['ws']
        st = cur.replace(hour=random.randint(0,23), minute=random.randint(0,59))
        dur = random.randint(2, 240); en = st + timedelta(minutes=dur)
        rs = random.choices(['SUCCEEDED','FAILED','CANCELLED','TIMED_OUT'], weights=[.88,.07,.03,.02])[0]
        tc = 'SUCCESS' if rs=='SUCCEEDED' else ('RUN_EXECUTION_ERROR' if rs=='FAILED' else ('USER_CANCELLED' if rs=='CANCELLED' else 'MAX_RUN_DURATION_EXCEEDED'))
        tr = random.choice(['SCHEDULED','MANUAL','RETRY','FILE_ARRIVAL'])
        jr_rows.append(
            f"('{ACCOUNT_ID}','{ws[0]}','{j['id']}','run-{jrc:08d}',"
            f"'{st.strftime('%Y-%m-%d %H:%M:%S')}','{en.strftime('%Y-%m-%d %H:%M:%S')}',"
            f"'{tr}','JOB_RUN','{rs}','{tc}')")
    cur += timedelta(days=1)

print(f'Generated {len(jr_rows)} job run rows. Inserting...')
for i, chunk in enumerate(chunk_list(jr_rows, 500)):
    run_sql(f'job_runs chunk {i+1}/{(len(jr_rows)//500)+1}', f'INSERT INTO workspace.mock_system_lakeflow.job_run_timeline VALUES\n' + ',\n'.join(chunk))

# Job tasks + task run timeline + pipelines (lightweight)
run_sql('drop job_tasks', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.job_tasks')
run_sql('create job_tasks', 'CREATE TABLE workspace.mock_system_lakeflow.job_tasks (account_id STRING,workspace_id STRING,job_id STRING,task_key STRING,change_time TIMESTAMP,delete_time TIMESTAMP)')
tk_rows = []
for j in JOBS[:200]:
    ws = j['ws']
    for t in range(random.randint(1,8)):
        tk_rows.append(f"('{ACCOUNT_ID}','{ws[0]}','{j['id']}','task-{t+1}','{(NOW-timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d %H:%M:%S')}',cast(null as TIMESTAMP))")
for i, chunk in enumerate(chunk_list(tk_rows, 500)):
    run_sql(f'tasks chunk {i+1}', f'INSERT INTO workspace.mock_system_lakeflow.job_tasks VALUES\n' + ',\n'.join(chunk))

run_sql('drop task_runs', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.job_task_run_timeline')
run_sql('create task_runs', 'CREATE TABLE workspace.mock_system_lakeflow.job_task_run_timeline (account_id STRING,workspace_id STRING,job_id STRING,run_id STRING,task_key STRING,period_start_time TIMESTAMP,period_end_time TIMESTAMP,result_state STRING,termination_code STRING)')

# Pipelines
run_sql('drop pipelines', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.pipelines')
run_sql('create pipelines', '''
CREATE TABLE workspace.mock_system_lakeflow.pipelines (
  account_id STRING, workspace_id STRING, pipeline_id STRING, name STRING,
  creator_user_name STRING, run_as_user_name STRING, channel STRING,
  edition STRING, created_time TIMESTAMP, change_time TIMESTAMP, delete_time TIMESTAMP
)
''')
pnames = ['CDC-Customer-Accounts','Fraud-Alerts-Stream','Transaction-ETL','Risk-Score-Pipeline',
          'AML-Detection','Market-Data-Ingest','Regulatory-Reporting','Customer-360-Build',
          'Credit-Score-Update','Payment-Processing','Loan-Origination','Card-Transaction-Stream',
          'KYC-Verification','Portfolio-Analytics','Trade-Settlement','Compliance-Audit-Trail',
          'Digital-Banking-Events','ATM-Transaction-Stream','Branch-Performance','Wealth-Portfolio-Sync']
p_rows = []
for i, pn in enumerate(pnames):
    ws = WORKSPACES[i % len(WORKSPACES)]; cr = random.choice(USERS)
    ed = random.choice(['CORE','PRO','ADVANCED'])
    p_rows.append(f"('{ACCOUNT_ID}','{ws[0]}','dlt-{i+1:03d}','{pn}','{sql_str(cr)}','{sql_str(cr)}','CURRENT','{ed}','{(NOW-timedelta(days=random.randint(100,900))).strftime('%Y-%m-%d %H:%M:%S')}','{(NOW-timedelta(days=random.randint(1,30))).strftime('%Y-%m-%d %H:%M:%S')}',cast(null as TIMESTAMP))")
run_sql('insert pipelines', f'INSERT INTO workspace.mock_system_lakeflow.pipelines VALUES\n' + ',\n'.join(p_rows))

run_sql('drop pipe_updates', 'DROP TABLE IF EXISTS workspace.mock_system_lakeflow.pipeline_update_timeline')
run_sql('create pipe_updates', 'CREATE TABLE workspace.mock_system_lakeflow.pipeline_update_timeline (account_id STRING,workspace_id STRING,pipeline_id STRING,update_id STRING,result_state STRING,period_start_time TIMESTAMP,period_end_time TIMESTAMP)')

print(f'✅ lakeflow loaded: {len(JOBS)} jobs, {len(jr_rows)} runs, {len(tk_rows)} tasks, {len(pnames)} pipelines')

In [ ]:
# ── Step 9: query.history ────────────────────────────────────────────
print('Generating query history...')
run_sql('drop query_history', 'DROP TABLE IF EXISTS workspace.mock_system_query.history')
run_sql('create query_history', '''
CREATE TABLE workspace.mock_system_query.history (
  statement_id STRING, executed_by STRING, executed_as STRING,
  compute STRUCT<warehouse_id: STRING>, statement_type STRING, statement_text STRING,
  start_time TIMESTAMP, total_duration_ms BIGINT, read_bytes BIGINT,
  read_rows BIGINT, error_message STRING
)
''')

random.seed(77)
queries_tmpl = [
    ('SELECT','SELECT * FROM transactions WHERE date >= current_date - 30'),
    ('SELECT','SELECT customer_id, SUM(amount) FROM payments GROUP BY 1'),
    ('SELECT','SELECT risk_score, COUNT(*) FROM credit_assessments GROUP BY 1'),
    ('INSERT','INSERT INTO daily_aggregates SELECT date, SUM(volume) FROM trades GROUP BY 1'),
    ('MERGE','MERGE INTO customer_360 USING staging_customers ON id = s_id'),
    ('SELECT','SELECT * FROM fraud_alerts WHERE score > 0.9 AND created_date = current_date'),
    ('SELECT','SELECT acct_type, AVG(balance) FROM accounts GROUP BY 1'),
    ('SELECT','SELECT COUNT(DISTINCT customer_id) FROM digital_banking_events'),
    ('SELECT','SELECT department, SUM(cost) FROM cost_allocation GROUP BY 1'),
]
q_rows = []
qc = 0
cur = THREE_YEARS_AGO
while cur < NOW:
    for _ in range(random.randint(100, 400)):
        qc += 1
        user = random.choice(USERS); wh = random.choice(WAREHOUSES)
        st_type, st_text = random.choice(queries_tmpl)
        start = cur.replace(hour=random.randint(7,21), minute=random.randint(0,59))
        dur = random.randint(50, 300000)
        he = random.random() < 0.02
        q_rows.append(
            f"('stmt-{qc:08d}','{sql_str(user)}','{sql_str(user)}',"
            f"named_struct('warehouse_id','{wh['id']}'),'{st_type}','{sql_str(st_text)}',"
            f"'{start.strftime('%Y-%m-%d %H:%M:%S')}',{dur},{random.randint(1000,10737418240)},"
            f"{random.randint(100,50000000)},{'cast(null as STRING)' if not he else repr('TABLE_OR_VIEW_NOT_FOUND')})")
    cur += timedelta(days=7)  # weekly sample

print(f'Generated {len(q_rows)} query rows. Inserting...')
for i, chunk in enumerate(chunk_list(q_rows, 500)):
    run_sql(f'queries chunk {i+1}', f'INSERT INTO workspace.mock_system_query.history VALUES\n' + ',\n'.join(chunk))
print(f'✅ query.history loaded: {len(q_rows)} rows')

In [ ]:
# ── Step 10: ai_gateway.usage (1.5 years, growing) ──────────────────
print('Generating AI gateway usage...')
run_sql('drop ai_gateway', 'DROP TABLE IF EXISTS workspace.mock_system_ai_gateway.usage')
run_sql('create ai_gateway', '''
CREATE TABLE workspace.mock_system_ai_gateway.usage (
  route_name STRING, event_time TIMESTAMP, total_token_count BIGINT,
  input_token_count BIGINT, output_token_count BIGINT,
  execution_duration_ms DOUBLE, requester STRING
)
''')

random.seed(99)
routes = ['gpt-4-turbo','claude-3-sonnet','databricks-dbrx','databricks-meta-llama-3',
          'databricks-mixtral','text-embedding-ada-002','databricks-bge-large']
ai_rows = []
ai_start = NOW - timedelta(days=540)
cur = ai_start
while cur < NOW:
    n = random.randint(50, 300)
    gf = 1 + ((cur - ai_start).days / 540) * 3
    n = int(n * gf)
    for _ in range(min(n, 500)):
        user = random.choice(USERS[:400]); route = random.choice(routes)
        et = cur.replace(hour=random.randint(8,20), minute=random.randint(0,59))
        inp = random.randint(100,8000); out = random.randint(50,4000)
        ai_rows.append(
            f"('{route}','{et.strftime('%Y-%m-%d %H:%M:%S')}',{inp+out},{inp},{out},"
            f"{random.uniform(100,5000):.1f},'{sql_str(user)}')")
    cur += timedelta(days=1)

print(f'Generated {len(ai_rows)} AI rows. Inserting...')
for i, chunk in enumerate(chunk_list(ai_rows, 500)):
    run_sql(f'ai chunk {i+1}', f'INSERT INTO workspace.mock_system_ai_gateway.usage VALUES\n' + ',\n'.join(chunk))
print(f'✅ ai_gateway.usage loaded: {len(ai_rows)} rows')

In [ ]:
# ── Step 11: serving, mlflow, lineage, storage, info_schema ─────────
print('Loading remaining tables...')

# ── served_entities
run_sql('drop served_entities', 'DROP TABLE IF EXISTS workspace.mock_system_serving.served_entities')
run_sql('create served_entities', 'CREATE TABLE workspace.mock_system_serving.served_entities (endpoint_name STRING,served_entity_name STRING,entity_type STRING,workspace_id STRING,change_time TIMESTAMP)')
endpoints = [('fraud-scorer','fraud-detection-v3','CUSTOM_MODEL'),('credit-model','credit-risk-xgb','CUSTOM_MODEL'),
    ('recommender','product-recommender-v2','CUSTOM_MODEL'),('nlp-classifier','doc-classifier-bert','CUSTOM_MODEL'),
    ('llm-gateway','databricks-meta-llama-3','FOUNDATION_MODEL'),('embedding-service','databricks-bge-large','FOUNDATION_MODEL'),
    ('aml-detector','aml-detection-ensemble','CUSTOM_MODEL'),('kyc-verifier','kyc-document-ocr','CUSTOM_MODEL')]
ep_rows = [f"('{n}','{e}','{t}','{random.choice(WORKSPACES)[0]}','{(NOW-timedelta(days=random.randint(30,300))).strftime('%Y-%m-%d %H:%M:%S')}')" for n,e,t in endpoints]
run_sql('insert served_entities', f'INSERT INTO workspace.mock_system_serving.served_entities VALUES\n' + ',\n'.join(ep_rows))

# ── endpoint_usage (1 year)
run_sql('drop endpoint_usage', 'DROP TABLE IF EXISTS workspace.mock_system_serving.endpoint_usage')
run_sql('create endpoint_usage', 'CREATE TABLE workspace.mock_system_serving.endpoint_usage (served_entity_name STRING,request_time TIMESTAMP,total_token_count BIGINT,status_code INT)')
random.seed(133)
eu_rows = []
for d_off in range(365):
    d = NOW - timedelta(days=d_off)
    for _,entity,etype in endpoints:
        for _ in range(min(random.randint(50,500), 100)):
            t = d.replace(hour=random.randint(0,23), minute=random.randint(0,59))
            tok = random.randint(100,5000) if 'FOUNDATION' in etype else random.randint(0,10)
            st = 200 if random.random()<0.97 else random.choice([400,500,503])
            eu_rows.append(f"('{entity}','{t.strftime('%Y-%m-%d %H:%M:%S')}',{tok},{st})")
print(f'Endpoint usage: {len(eu_rows)} rows')
for i, chunk in enumerate(chunk_list(eu_rows, 500)):
    run_sql(f'ep_usage chunk {i+1}', f'INSERT INTO workspace.mock_system_serving.endpoint_usage VALUES\n' + ',\n'.join(chunk))

# ── MLflow experiments
run_sql('drop experiments', 'DROP TABLE IF EXISTS workspace.mock_system_mlflow.experiments_latest')
run_sql('create experiments', 'CREATE TABLE workspace.mock_system_mlflow.experiments_latest (experiment_id STRING,name STRING,lifecycle_stage STRING,creation_time BIGINT,last_update_time BIGINT)')
ml_exps = ['Fraud-Detection-V3','Credit-Risk-XGBoost','Customer-Churn-Prediction','Product-Recommender',
    'NLP-Document-Classifier','AML-Ensemble','Transaction-Anomaly','Loan-Default-Prediction',
    'Sentiment-Analysis','Market-Risk-VAR','Portfolio-Optimization','KYC-OCR-Model',
    'Card-Fraud-RealTime','Customer-LTV','Cross-Sell-Propensity','Branch-Demand-Forecast',
    'ATM-Cash-Optimization','Interest-Rate-Model','Mortgage-Prepayment','Collections-Priority']
exp_rows = []
for i, en in enumerate(ml_exps):
    ct = int((NOW-timedelta(days=random.randint(90,900))).timestamp()*1000)
    ut = int((NOW-timedelta(days=random.randint(1,60))).timestamp()*1000)
    stg = 'active' if random.random()<0.85 else 'deleted'
    exp_rows.append(f"('exp-{i+1:03d}','{en}','{stg}',{ct},{ut})")
run_sql('insert experiments', f'INSERT INTO workspace.mock_system_mlflow.experiments_latest VALUES\n' + ',\n'.join(exp_rows))

# ── MLflow runs
run_sql('drop ml_runs', 'DROP TABLE IF EXISTS workspace.mock_system_mlflow.runs_latest')
run_sql('create ml_runs', 'CREATE TABLE workspace.mock_system_mlflow.runs_latest (experiment_id STRING,run_id STRING,status STRING,start_time TIMESTAMP,end_time TIMESTAMP,user_id STRING)')
random.seed(144)
mr_rows = []
for i in range(len(ml_exps)):
    for r in range(random.randint(20,200)):
        st = NOW - timedelta(days=random.randint(1,365), hours=random.randint(0,23))
        en = st + timedelta(minutes=random.randint(5,480))
        status = random.choices(['FINISHED','FAILED','RUNNING','KILLED'],weights=[.75,.12,.08,.05])[0]
        mr_rows.append(f"('exp-{i+1:03d}','mlrun-{i+1:03d}-{r+1:04d}','{status}','{st.strftime('%Y-%m-%d %H:%M:%S')}','{en.strftime('%Y-%m-%d %H:%M:%S')}','{sql_str(random.choice(USERS[:300]))}')")
for i, chunk in enumerate(chunk_list(mr_rows, 500)):
    run_sql(f'ml_runs chunk {i+1}', f'INSERT INTO workspace.mock_system_mlflow.runs_latest VALUES\n' + ',\n'.join(chunk))

# ── Registered models
run_sql('drop models', 'DROP TABLE IF EXISTS workspace.mock_system_mlflow.registered_models_latest')
run_sql('create models', 'CREATE TABLE workspace.mock_system_mlflow.registered_models_latest (name STRING,creation_timestamp BIGINT,last_updated_timestamp BIGINT,user_id STRING)')
reg_models = ['fraud-detection-v3','credit-risk-xgb','customer-churn-model','product-recommender-v2',
    'doc-classifier-bert','aml-detection-ensemble','transaction-anomaly-v1','loan-default-pred',
    'sentiment-analyzer','market-risk-var','portfolio-optimizer','kyc-document-ocr']
rm_rows = [f"('{m}',{int((NOW-timedelta(days=random.randint(30,600))).timestamp()*1000)},{int((NOW-timedelta(days=random.randint(1,30))).timestamp()*1000)},'{sql_str(random.choice(USERS[:200]))}')" for m in reg_models]
run_sql('insert models', f'INSERT INTO workspace.mock_system_mlflow.registered_models_latest VALUES\n' + ',\n'.join(rm_rows))

print(f'✅ serving loaded: {len(endpoints)} endpoints, {len(eu_rows)} usage rows')
print(f'✅ mlflow loaded: {len(ml_exps)} experiments, {len(mr_rows)} runs, {len(reg_models)} models')

In [ ]:
# ── Step 12: lineage, storage, info_schema, marketplace ─────────────
print('Loading lineage, storage, and remaining tables...')

# Table lineage
run_sql('drop tbl_lineage', 'DROP TABLE IF EXISTS workspace.mock_system_access.table_lineage')
run_sql('create tbl_lineage', 'CREATE TABLE workspace.mock_system_access.table_lineage (source_table_catalog STRING,source_table_schema STRING,source_table_name STRING,target_table_catalog STRING,target_table_schema STRING,target_table_name STRING,event_time TIMESTAMP)')
lin_pairs = [('raw_transactions','bronze_transactions'),('bronze_transactions','silver_transactions'),
    ('silver_transactions','gold_transaction_summary'),('raw_customers','bronze_customers'),
    ('bronze_customers','silver_customer_360'),('silver_customer_360','gold_customer_analytics'),
    ('raw_market_data','bronze_market_data'),('bronze_market_data','silver_risk_metrics'),
    ('silver_risk_metrics','gold_risk_dashboard'),('raw_fraud_events','bronze_fraud_alerts'),
    ('bronze_fraud_alerts','silver_fraud_scores'),('silver_fraud_scores','gold_fraud_summary'),
    ('raw_loan_applications','bronze_loan_data'),('bronze_loan_data','silver_credit_assessment'),
    ('silver_credit_assessment','gold_lending_analytics')]
lin_rows = []
for src, tgt in lin_pairs:
    for d in range(30):
        dt = (NOW - timedelta(days=d)).strftime('%Y-%m-%d')
        lin_rows.append(f"('workspace','banking_data','{src}','workspace','banking_data','{tgt}','{dt} 08:00:00')")
run_sql('insert lineage', f'INSERT INTO workspace.mock_system_access.table_lineage VALUES\n' + ',\n'.join(lin_rows))

# Column lineage
run_sql('drop col_lineage', 'DROP TABLE IF EXISTS workspace.mock_system_access.column_lineage')
run_sql('create col_lineage', 'CREATE TABLE workspace.mock_system_access.column_lineage (source_table_catalog STRING,source_table_schema STRING,source_table_name STRING,source_column_name STRING,target_table_catalog STRING,target_table_schema STRING,target_table_name STRING,target_column_name STRING,event_time TIMESTAMP)')
col_pairs = [('raw_transactions','amount','silver_transactions','total_amount'),
    ('raw_transactions','customer_id','silver_transactions','customer_id'),
    ('raw_customers','email','silver_customer_360','contact_email'),
    ('raw_fraud_events','score','silver_fraud_scores','risk_score')]
cl_rows2 = [f"('workspace','banking_data','{s}','{sc}','workspace','banking_data','{t}','{tc}','{(NOW-timedelta(days=1)).strftime('%Y-%m-%d')} 08:00:00')" for s,sc,t,tc in col_pairs]
run_sql('insert col_lineage', f'INSERT INTO workspace.mock_system_access.column_lineage VALUES\n' + ',\n'.join(cl_rows2))

# Storage optimization
run_sql('drop storage_opt', 'DROP TABLE IF EXISTS workspace.mock_system_storage.predictive_optimization_operations_history')
run_sql('create storage_opt', 'CREATE TABLE workspace.mock_system_storage.predictive_optimization_operations_history (catalog_name STRING,schema_name STRING,table_name STRING,operation_type STRING,operation_status STRING,start_time TIMESTAMP,end_time TIMESTAMP,operation_metrics MAP<STRING,STRING>)')
st_tables = ['transactions','customers','accounts','payments','loans','credit_scores','market_data','trade_history','fraud_alerts']
st_rows = []
for day in range(90):
    d = NOW - timedelta(days=day)
    for tbl in random.sample(st_tables, random.randint(2,6)):
        op = random.choice(['OPTIMIZE','VACUUM','ZORDER'])
        sts = 'SUCCEEDED' if random.random()<0.92 else 'FAILED'
        s = d.replace(hour=2, minute=random.randint(0,59))
        e = s + timedelta(minutes=random.randint(3,45))
        st_rows.append(f"('workspace','banking_data','{tbl}','{op}','{sts}','{s.strftime('%Y-%m-%d %H:%M:%S')}','{e.strftime('%Y-%m-%d %H:%M:%S')}',map('files_removed','{random.randint(1,500)}','bytes_removed','{random.randint(10000,50000000000)}'))")
run_sql('insert storage', f'INSERT INTO workspace.mock_system_storage.predictive_optimization_operations_history VALUES\n' + ',\n'.join(st_rows))

# Information schema
run_sql('drop info_tables', 'DROP TABLE IF EXISTS workspace.mock_system_information_schema.tables')
run_sql('create info_tables', 'CREATE TABLE workspace.mock_system_information_schema.tables (table_catalog STRING,table_schema STRING,table_name STRING,table_type STRING,created TIMESTAMP)')
info_rows = [f"('workspace','banking_data','{t}','MANAGED','{(NOW-timedelta(days=900)).strftime('%Y-%m-%d %H:%M:%S')}')" for t in st_tables]
run_sql('insert info_tables', f'INSERT INTO workspace.mock_system_information_schema.tables VALUES\n' + ',\n'.join(info_rows))

# Marketplace
run_sql('drop mkt_listings', 'DROP TABLE IF EXISTS workspace.mock_system_marketplace.listings')
run_sql('create mkt_listings', 'CREATE TABLE workspace.mock_system_marketplace.listings (listing_id STRING,listing_name STRING,provider STRING,category STRING,status STRING,created_time TIMESTAMP)')
listings = [('Market Risk Data Feed','Bloomberg','DATA'),('Fraud Intelligence','LexisNexis','DATA'),
    ('Credit Bureau Scores','Experian','DATA'),('Economic Indicators','Federal Reserve','DATA'),
    ('Geospatial Banking Data','SafeGraph','DATA')]
mkt_rows = [f"('mkt-{i+1:03d}','{n}','{p}','{c}','ACTIVE','{(NOW-timedelta(days=random.randint(90,700))).strftime('%Y-%m-%d %H:%M:%S')}')" for i,(n,p,c) in enumerate(listings)]
run_sql('insert mkt', f'INSERT INTO workspace.mock_system_marketplace.listings VALUES\n' + ',\n'.join(mkt_rows))

run_sql('drop mkt_access', 'DROP TABLE IF EXISTS workspace.mock_system_marketplace.listing_access')
run_sql('create mkt_access', 'CREATE TABLE workspace.mock_system_marketplace.listing_access (listing_id STRING,consumer_id STRING,access_type STRING,access_time TIMESTAMP)')
run_sql('drop mkt_funnel', 'DROP TABLE IF EXISTS workspace.mock_system_marketplace.listing_funnel_events')
run_sql('create mkt_funnel', 'CREATE TABLE workspace.mock_system_marketplace.listing_funnel_events (listing_id STRING,event_type STRING,event_time TIMESTAMP,consumer STRING)')

# Networking / Lakeview / Dashboards (empty structure)
for schema_tbl, ddl in [
    ('networking.private_endpoint_rules', 'account_id STRING,workspace_id STRING,rule_name STRING,resource_type STRING,status STRING,created_time TIMESTAMP'),
    ('networking.firewall_rules', 'account_id STRING,workspace_id STRING,rule_name STRING,cidr_block STRING,status STRING,created_time TIMESTAMP'),
    ('lakeview.dashboards', 'dashboard_id STRING,name STRING,creator STRING,created_time TIMESTAMP,updated_time TIMESTAMP'),
    ('lakeview.dashboard_usage', 'dashboard_id STRING,user_email STRING,view_time TIMESTAMP'),
    ('dashboards.dashboards', 'dashboard_id STRING,name STRING,creator STRING,created_time TIMESTAMP,updated_time TIMESTAMP'),
    ('dashboards.dashboard_usage', 'dashboard_id STRING,user_email STRING,view_time TIMESTAMP'),
]:
    full = f'workspace.mock_system_{schema_tbl}'
    run_sql(f'drop {schema_tbl}', f'DROP TABLE IF EXISTS {full}')
    run_sql(f'create {schema_tbl}', f'CREATE TABLE {full} ({ddl})')

print(f'✅ All remaining tables loaded!')
print(f'   Lineage: {len(lin_pairs)} table pairs × 30 days')
print(f'   Storage ops: {len(st_rows)} optimization records')
print(f'   Marketplace: {len(listings)} listings')

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────
print('\n' + '='*70)
print('🎉 ENTERPRISE MOCK DATA LOAD COMPLETE')
print('='*70)
print(f'''
📊 Data Summary:
   Users:              {len(USERS):,}
   Departments:        {len(DEPARTMENTS)}
   Teams:              {len(TEAMS)}
   Workspaces:         {len(WORKSPACES)}
   Clusters:           {len(CLUSTERS):,}
   Warehouses:         {len(WAREHOUSES)}
   Jobs:               {len(JOBS):,}
   DLT Pipelines:      {len(pnames)}
   ML Experiments:     {len(ml_exps)}
   Registered Models:  {len(reg_models)}
   Serving Endpoints:  {len(endpoints)}
   
💰 Billing:
   Year 1 target:      ~$3.5M
   Year 2 target:      ~$4.5M
   Year 3 target:      ~$5.5M (forecast)
   Date range:         {THREE_YEARS_AGO.strftime("%Y-%m-%d")} → {NOW.strftime("%Y-%m-%d")}

📋 Tables loaded in workspace.mock_system_* schemas:
   billing.usage, billing.list_prices
   access.audit, access.workspaces_latest
   access.table_lineage, access.column_lineage
   compute.clusters, compute.warehouses, compute.node_timeline
   compute.cluster_events, compute.warehouse_events
   lakeflow.jobs, lakeflow.job_run_timeline, lakeflow.job_tasks
   lakeflow.job_task_run_timeline, lakeflow.pipelines, lakeflow.pipeline_update_timeline
   query.history
   ai_gateway.usage
   serving.served_entities, serving.endpoint_usage
   mlflow.experiments_latest, mlflow.runs_latest, mlflow.registered_models_latest
   storage.predictive_optimization_operations_history
   information_schema.tables
   marketplace.listings, marketplace.listing_access, marketplace.listing_funnel_events
   + networking, lakeview, dashboards tables
''')